In [1]:
import pandas as pd
df = pd.read_csv("data.csv")
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [2]:
df["Review Rating"] = df.groupby("Category")["Review Rating"].transform(lambda x: x.fillna(x.mean()))
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [3]:
df.columns = df.columns.str.lower().str.replace(" ","_")
df = df.rename(columns={'purchase_amount_(usd)' : 'purchase_amount'})
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

In [4]:
# create age group using qcut
age_type = ['young','adult', 'middle-aged', 'senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=age_type)
df["age_group"].head(10)

0    middle-aged
1          young
2    middle-aged
3          young
4    middle-aged
5    middle-aged
6         senior
7          young
8          young
9    middle-aged
Name: age_group, dtype: category
Categories (4, str): ['young' < 'adult' < 'middle-aged' < 'senior']

In [5]:
# create columns purchase_frequency_days
frequency_mapping = {
    'fortnightly': 14,
    'monthly': 30,
    'weekly': 7,
    'quarterly': 90,
    'bi-weekly': 14,
    'annually': 365,
    'every 3 months': 90
}
df['frequency_of_purchases'] = df['frequency_of_purchases'].str.lower()
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)
df[['purchase_frequency_days', 'frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,fortnightly
1,14,fortnightly
2,7,weekly
3,7,weekly
4,365,annually
5,7,weekly
6,90,quarterly
7,7,weekly
8,365,annually
9,90,quarterly


In [6]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [7]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="rakshithvinay56",  # fix this
    database="data_analysis"
)

print("Connected successfully!")

Connected successfully!


In [8]:
!pip install sqlalchemy pymysql


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from sqlalchemy import create_engine

# MySQL connection
username = "root"
password = "rakshithvinay56"
host = "localhost"
port = "3306"
database = "data_analysis"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

# # Sample DataFrame
# data = {
#     "name": ["Rakshith", "John", "Amit"],
#     "age": [22, 25, 23]
# }
# df = pd.DataFrame(data)

# Write to MySQL
table_name = "mytable"
df.to_sql(table_name, engine, if_exists="replace", index=False)

# Read from MySQL
result = pd.read_sql("SELECT * FROM mytable LIMIT 5;", engine)
print(result)

   customer_id  age gender item_purchased  category  purchase_amount  \
0            1   55   Male         Blouse  Clothing               53   
1            2   19   Male        Sweater  Clothing               64   
2            3   50   Male          Jeans  Clothing               73   
3            4   21   Male        Sandals  Footwear               90   
4            5   45   Male         Blouse  Clothing               49   

        location size      color  season  review_rating subscription_status  \
0       Kentucky    L       Gray  Winter            3.1                 Yes   
1          Maine    L     Maroon  Winter            3.1                 Yes   
2  Massachusetts    S     Maroon  Spring            3.1                 Yes   
3   Rhode Island    M     Maroon  Spring            3.5                 Yes   
4         Oregon    M  Turquoise  Spring            2.7                 Yes   

   shipping_type discount_applied promo_code_used  previous_purchases  \
0        Express   